In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import mean_squared_error

color_pal = sns.color_palette()  # define color palette
plt.style.use('fivethirtyeight')

#### Outline of Problem
- Outlier analysis
- Forecasting horizon explained
- Time Series Cross-Validation
- Log Features
- Predicting the Future

In [0]:
filepath = "C:\\Users\\dkhan\\Desktop\\Python\\Machine Learning\\Time_Series\\pjme_hourly.csv"
df = pd.read_csv(filepath)
df = df.set_index('Datetime')
df.index = pd.to_datetime(df.index)

In [0]:
df.head()

In [0]:
df.plot(style='.', 
        figsize=(15,5), 
        color=color_pal[0], 
        title='PJME energy use in MW')
plt.show()

In [0]:
# in region between 2012 to 2014, there is a data show very low value. The model will learn from that
# data. If you remove the outlier make sure it has not legitimate role to data.

#### Outlier Removal and Histogram

In [0]:
df['PJME_MW'].plot(kind='hist', bins=500)

In [0]:
df.query('PJME_MW<20000').plot(figsize=(15,5), style='.')

In [0]:
# The values in the region between 2012 and 2014 are not legitimate. Let's go little less than 20000.

In [0]:
df.query('PJME_MW < 19000')\
.plot(style='.',
      figsize=(15,5),
      color=color_pal[5],
      title='Outliers')

In [0]:
df.query('PJME_MW > 19000').copy()

#### Reviewing: Train Test Split

In [0]:
train = df.loc[df.index<'01-01-2015']
test = df.loc[df.index>='01-01-2015']

fig, ax = plt.subplots(figsize=(15,5))
train.plot(ax=ax, label= 'training set', title='Train_Test_Split')
test.plot(ax=ax, label='test set')
ax.axvline('01-01-2015', color='black', ls='--')
ax.legend(['training_set', 'test_set'])
plt.show()
plt.show()

#### Time Series Cross Validation

In [0]:
from sklearn.model_selection import TimeSeriesSplit


tss = TimeSeriesSplit(n_splits=5, test_size=24*365*1, gap=24)    # Look at timeseriessplit in sklearn.
                    # test_size for 1 year, gap 24 for training and validation set in each time
df = df.sort_index()

In [0]:
for train_idx, val_idx in tss.split(df):
    break

In [0]:
train_idx

In [0]:
val_idx

In [0]:
fig, axs = plt.subplots(5, 1, figsize=(15, 15),
                       sharex=True)
fold = 0
for train_idx, val_idx in tss.split(df):
    train = df.iloc[train_idx]
    test = df.iloc[val_idx]
    train['PJME_MW'].plot(ax=axs[fold],
                          label='Training Set',
                          title = f'Data Train Test Split Fold{fold}')
    test['PJME_MW'].plot(ax=axs[fold],
                          label='Test Set')
    axs[fold].axvline(test.index.min(), color='black', ls='--')
    fold +=1
plt.show()

#### Forecasting Horizon Explained

In [0]:
def create_features(df):
    '''
    create time series features based on time series index.
    '''
    df =df.copy()
    df['hour']=df.index.hour 
    df['dayofweek']=df.index.day_of_week
    df['quarter']=df.index.quarter 
    df['month']=df.index.month
    df['year']=df.index.year
    df['dayofyear']=df.index.dayofyear
    return df

df = create_features(df)

#### 3. Lag Features

In [0]:
# what was the target(x) days in the past.

In [0]:
target_map = df['PJME_MW'].to_dict()

In [0]:
def add_lags(df):
    df['lag1'] = (df.index-pd.Timedelta('364 days')).map(target_map)   # not 365 beacause 364/7
    df['lag2'] = (df.index-pd.Timedelta('728 days')).map(target_map) 
    df['lag3'] = (df.index-pd.Timedelta('1092 days')).map(target_map)
    return df

In [0]:
df = add_lags(df)

#### Train Using Cross Validation

In [0]:
df.head()

In [0]:
df.tail()

In [0]:
tss = TimeSeriesSplit(n_splits=5, test_size=24*365*1, gap=24) 
df = df.sort_index()

folder = 0
preds = []
scores = []

for train_idx, val_idx in tss.split(df):
    train = df.iloc[train_idx]
    test = df.iloc[val_idx]
    train = create_features(train)
    test = create_features(test)
    FEATURES = ['hour', 'dayofweek', 'quarter', 'month', 'year','dayofyear']
    TARGET = 'PJME_MW'
    X_train = train[FEATURES]
    y_train = train[TARGET]
    
    X_test = test[FEATURES]
    y_test = test[TARGET]

    reg =xgb.XGBRegressor(base_score=0.5, booster='gbtree',
                          n_estimators=1000,
                          early_stopping_rounds=50,
                          objective='reg:linear',
                          max_depth=3,
                          learning_rate=0.01) # we use regression test

    reg.fit(X_train, y_train,
            eval_set=[(X_train, y_train), (X_test, y_test)],
            verbose=100)  # gives every 100
    y_pred = reg.predict(X_test)
    preds.append(y_pred)
    score = np.sqrt(mean_squared_error(y_test, y_pred))
    scores.append(score)

In [0]:
scores

In [0]:
print(f'Scores across folds {np.mean(scores):0.4f}') # evaluating by mean method for cross validation
print(f'Fold scores:{scores}')

#### 4. Predicting the Future

- Retraining all data
- To predict future we need an empty dataframe for future data ranges
- Run those date through our feature creation code + lag creation 

In [0]:
# Retarin all data
df = create_features(df)

FEATURES = ['hour', 'dayofweek', 'quarter', 'month', 'year','dayofyear', 
            'lag1', 'lag2', 'lag3']
TARGET = 'PJME_MW'
X_all = df[FEATURES]
y_all = df[TARGET]

reg =xgb.XGBRegressor(base_score=0.5, booster='gbtree',
                      n_estimators=500,
                      objective='reg:linear',
                      max_depth=3,
                      learning_rate=0.01) # we use regression test

reg.fit(X_all, y_all,
        eval_set=[(X_all, y_all)],
        verbose=100)  # gives every 100


In [0]:
df.index.max() # Training goes upto max value

In [0]:
# Create Future DataFrame
future = pd.date_range('2018-08-03', '2019-08-01', freq='1h') 
future_df = pd.DataFrame(index=future)
future_df['isFuture']= True
df['isFuture']=False
df_and_future = pd.concat([df, future_df])

In [0]:
# Above,go upto first day of that month 
# as pour last date so this is end date of our data range. We have to predict hourly, so freq=1h

In [0]:
df_and_future = create_features(df_and_future)
df_and_future = add_lags(df_and_future)

In [0]:
df_and_future = pd.concat([df, future_df])
df_and_future = create_features(df_and_future)
df_and_future = add_lags(df_and_future)

In [0]:
future_with_features = df_and_future.query('isFuture').copy()

In [0]:
future_with_features

#### Predict the Future

In [0]:
future_with_features['pred']=reg.predict(future_with_features[FEATURES])

In [0]:
future_with_features['pred'].plot(figsize=(15,5),
                                  color=color_pal[4],
                                  ms=1, 
                                  lw=1, 
                                  title='Future Predictions') 
                                  

#### Save model for Later

In [0]:
# save model 1
reg.save_model('model.jason')

In [0]:
!dir

In [0]:
reg_new = xgb.XGBRegressor()
reg_new.load_model('model.jason')

future_with_features['pred']=reg_new.predict(future_with_features[FEATURES]) # change reg to reg_new
future_with_features['pred'].plot(figsize=(15,5),
                                  color=color_pal[4],
                                  ms=1, 
                                  lw=1, 
                                  title='Future Predictions') 